# MRI/CT → ReVOS 完整数据集制作流程（train / train_eval 分离版）

本 notebook 将 **3D MRI/CT 医学图像** 当作**视频流**（切片即帧），制作为 DTOS Stage-2 所需的格式，
并按 `TRAIN_RATIO` 划分为 **train** 和 **train_eval** 两个完全独立的子目录，每个子目录含：

```
OUTPUT_REVOS_DIR/
├── train/
│   ├── JPEGImages/          # 帧图像
│   ├── Annotations/         # 索引 PNG（像素值 = obj_id）
│   ├── meta.json            # generate_mask_dict 所需的 meta
│   ├── meta_expressions.json# DTOS RVOSDataset 所需
│   └── mask_dict.pkl        # 每条 expression 的 RLE mask 列表（100 帧）
└── train_eval/              # 评估集（目录名含 'train' 以保证 get_dataset_mode 返回 'train'）
    ├── JPEGImages/
    ├── Annotations/
    ├── meta.json
    ├── meta_expressions.json
    └── mask_dict.pkl
```

**⚠️ 为什么评估目录命名为 `train_eval` 而非 `test`？**

`get_dataset_mode('youtubervos', path)` 按路径字符串匹配模式：
- 含 `'train'` → mode='train'（读取 meta.json，正确分配 anno_id）
- 含 `'test'`  → mode='test'（硬编码 anno_id=['-1'] → KeyError: '-1'）

我们的评估集有真实 mask，必须走 train 分支，故目录名含 `'train'`。

**mask_dict.pkl 格式**（与 `generate_mask_dict.py` 完全一致）：
- key: `str(anno_id)`，从 0 开始，按 `sorted(videos) → sorted(expressions)` 顺序编号（每个 split 独立计数）
- value: 长度为 100 的列表，每项为 COCO-RLE dict 或 `None`（无前景时）

**流程：**
1. 加载 M3D-Seg 样本列表
2. 按 TRAIN_RATIO 随机划分
3. 主循环：按 split 写入 JPEGImages、Annotations，内联构建 mask_dict
4. 保存每个 split 的 meta.json / meta_expressions.json / mask_dict.pkl
5. 快速校验

## 0. 依赖与导入

In [55]:
import os
import re
import json
import pickle
import random
import numpy as np
from PIL import Image
from tqdm import tqdm
from scipy import sparse
import ast

try:
    import pycocotools.mask as maskUtils
except ImportError:
    maskUtils = None

print('Imports OK')

Imports OK


## 1. 配置

In [57]:
# ── 路径 ──────────────────────────────────────────────────────────────────────
M3D_SEG_ROOT      = '/BDSZ6/private/user/yxd/data/M3D/M3D-Seg/'
PATH_BASE         = os.path.join(M3D_SEG_ROOT, 'M3D_Seg')   # 存放 {dc}/{dc}.json 和 .npy/.npz
OUTPUT_REVOS_DIR  = '/BDSZ6/private/user/yxd/data/M3D/data_all'  # 根输出目录
TRAIN_DIR         = os.path.join(OUTPUT_REVOS_DIR, 'train')
# 注意：目录名必须含 'train'，否则 get_dataset_mode('youtubervos', path) 会匹配到 'test'
# 并进入 load_refytvos_json 的 else 分支，将所有 anno_id 硬编码为 '-1'，
# 导致 rvos.py 中 mask_dict['-1'] KeyError。
TEST_DIR          = os.path.join(OUTPUT_REVOS_DIR, 'train_eval')

TERM_DICTIONARY_PATH = '/BDSZ6/private/user/yxd/data/M3D/M3D-Seg/term_dictionary.json'
LABEL2NAME_PATH      = None   # 若无则设为 None

# ── 参数 ──────────────────────────────────────────────────────────────────────
N_SLICES     = 100
AXIS         = 'axial'
SAMPLING     = 'uniform'
TRAIN_RATIO  = 0.8
SEED         = 42
WINDOW_LEVEL = -600
WINDOW_WIDTH = 1600

DATASET_CODES = [
# '0006'
# '0018','0019','0020','0022'
# '0002'
    '0000','0001','0002','0004','0005','0006','0007','0008','0009',
    '00010','0011','0012','0013','0014','0015','00016','0017','0018','0019','0020','0021','0022'
]

random.seed(SEED)
print(f'TRAIN_DIR : {TRAIN_DIR}')
print(f'TEST_DIR  : {TEST_DIR}')
print(f'Window    : WL={WINDOW_LEVEL}, WW={WINDOW_WIDTH}')


TRAIN_DIR : /BDSZ6/private/user/yxd/data/M3D/data_all/train
TEST_DIR  : /BDSZ6/private/user/yxd/data/M3D/data_all/train_eval


## 2. 辅助函数

In [ ]:
def load_3d_medical_image(
    image_path, n_slices=100, axis='axial', sampling='uniform',
    slice_range=None, normalize=True, window_level=-600, window_width=1600,
):
    """加载 .npy 3D 医学图像，返回 (PIL RGB 列表, 切片索引列表)。

    对 MRI 切片先应用固定窗宽窗位，再映射到 0-255。
    每张切片直接将单通道灰度值复制到 R/G/B 三个通道（无相邻帧合成）。
    """
    if image_path.endswith('.npy'):
        img = np.load(image_path)
    else:
        raise ValueError(f'仅支持 .npy 格式: {image_path}')
    if len(img.shape) != 3:
        if len(img.shape) == 4 and img.shape[0] == 1:
            img = img.squeeze(0)
        else:
            raise ValueError('需要 3D 数据')

    if window_width <= 0:
        raise ValueError(f'window_width 必须 > 0，当前为 {window_width}')

    ax_map = {'axial': 2, 'coronal': 0, 'sagittal': 1}
    total_slices = img.shape[ax_map[axis]]
    start, end = (0, total_slices) if slice_range is None else slice_range
    start, end = max(0, start), min(end, total_slices)
    available = end - start

    if available < n_slices:
        base = np.arange(start, end)
        repeat = n_slices // available
        remainder = n_slices % available
        indices = np.repeat(base, repeat).tolist() + base[:remainder].tolist()
        indices = indices[:n_slices]
    elif sampling == 'uniform':
        indices = np.linspace(start, end - 1, n_slices, dtype=int).tolist()
    elif sampling == 'random':
        indices = sorted(random.sample(range(start, end), n_slices))
    else:
        indices = np.linspace(start, end - 1, n_slices, dtype=int).tolist()

    def _get_slice(idx):
        actual = max(0, min(int(idx), total_slices - 1))
        if axis == 'axial':
            return img[:, :, actual]
        elif axis == 'coronal':
            return img[actual, :, :]
        else:
            return img[:, actual, :]

    def apply_window(ch):
        ch = np.asarray(ch, dtype=np.float32)
        lower = float(window_level) - float(window_width) / 2.0
        upper = float(window_level) + float(window_width) / 2.0
        ch = np.clip(ch, lower, upper)
        if normalize:
            ch = (ch - lower) / (upper - lower) * 255.0
        return np.clip(ch, 0, 255).astype(np.uint8)

    slices_list = []
    for idx in indices:
        gray = apply_window(_get_slice(idx))
        rgb = np.stack([gray, gray, gray], axis=-1)
        slices_list.append(Image.fromarray(rgb, mode='RGB'))
    return slices_list, indices


In [59]:
# ── Mask 加载 / 切片 ───────────────────────────────────────────────────────────

def get_mask_path_from_image_path(path_base, image_rel):
    image_dir = os.path.join(path_base, os.path.dirname(image_rel))
    if not os.path.isdir(image_dir):
        return None
    for fname in sorted(os.listdir(image_dir)):
        if fname.startswith('mask_') and fname.endswith('.npz'):
            return os.path.join(image_dir, fname)
    return None


def load_mask_npz(mask_path):
    """返回 (gt_array [C,H,W,D] uint8, depth)。"""
    if not mask_path or not os.path.isfile(mask_path):
        raise ValueError(f'Invalid mask path: {mask_path}')
    allmatrix_sp = sparse.load_npz(mask_path)
    gt_shape = ast.literal_eval(mask_path.split('.')[-2].split('_')[-1])
    gt_array = allmatrix_sp.toarray().reshape(gt_shape)
    gt_array = (gt_array != 0).astype(np.uint8)
    if gt_array.ndim != 4:
        raise ValueError(f'期望 4D，得到 {gt_array.ndim}D: {mask_path}')
    return gt_array, gt_array.shape[3]


def mask_slice_at_z(gt_array, mask_idx, z):
    z = min(max(0, z), gt_array.shape[3] - 1)
    return gt_array[mask_idx, :, :, z]  # uint8 {0,1}


def bbox_from_mask(mask):
    """返回单个包围整张 mask 的 bbox: (x1, y1, x2, y2)；空 mask 返回 None。"""
    if mask is None:
        return None
    mask = np.asarray(mask) > 0
    if not np.any(mask):
        return None
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    y1, y2 = np.where(rows)[0][[0, -1]]
    x1, x2 = np.where(cols)[0][[0, -1]]
    return int(x1), int(y1), int(x2), int(y2)


def mask_to_bbox_mask(mask):
    """把原始 mask 转成单个 bbox mask，保持下游 PNG/RLE 接口不变。"""
    bbox = bbox_from_mask(mask)
    bbox_mask = np.zeros_like(mask, dtype=np.uint8)
    if bbox is None:
        return bbox_mask
    x1, y1, x2, y2 = bbox
    bbox_mask[y1:y2 + 1, x1:x2 + 1] = 1
    return bbox_mask


# ── RLE 编码 ──────────────────────────────────────────────────────────────────

def rle_encode(mask):
    """二值 mask (H,W) → COCO RLE dict（空 mask 返回 None）。"""
    if maskUtils is None:
        raise ImportError('需要安装 pycocotools')
    if not np.any(mask):
        return None
    rle = maskUtils.encode(np.asfortranarray(mask.astype(np.uint8)))
    if isinstance(rle.get('counts'), bytes):
        rle['counts'] = rle['counts'].decode('utf-8')
    return rle


def rle_list_from_gt_array(gt_array, mask_idx, slice_indices):
    """
    从 gt_array 中提取单个 organ 在给定切片序列上的 bbox-mask RLE 列表。

    - 先把原始 mask 转成单个 enclosing bbox mask
    - 空帧返回 None
    - 列表长度 == len(slice_indices)（即 N_SLICES = 100）
    """
    masks = []
    for z in slice_indices:
        m = mask_to_bbox_mask(mask_slice_at_z(gt_array, mask_idx, z))
        masks.append(rle_encode(m))  # None if empty
    return masks


# ── Annotation PNG 写入 ───────────────────────────────────────────────────────

def save_indexed_annotation_frame(masks_per_frame, output_path, height, width):
    """masks_per_frame: [(obj_id, mask_2d), ...] → 索引 PNG（L 模式，像素值=obj_id）。

    ⚠️  必须用 mode='L'（灰度），不能用 mode='P'（调色板）。
    PIL 调色板 PNG 在没有显式设置恒等调色板的情况下，读回时 np.array() 会把
    像素值 2 映射为调色板颜色而非索引值，导致 obj_id ≥ 2 的 mask 全部丢失。
    mode='L' 直接把 uint8 像素值原样写入文件，读回后与写入完全一致。
    """
    indexed = np.zeros((height, width), dtype=np.uint8)
    for obj_id, mask_2d in masks_per_frame:
        if mask_2d is None or not np.any(mask_2d):
            continue
        np.copyto(indexed, np.uint8(obj_id), where=(mask_2d > 0))
    Image.fromarray(indexed, mode='L').save(output_path, format='PNG')


print('Helper functions defined.')


Helper functions defined.


In [60]:
# ── dataset JSON 缓存 ─────────────────────────────────────────────────────────
_dataset_json_cache = {}

def load_dataset_labels(path_base, dataset_code):
    if dataset_code in _dataset_json_cache:
        return _dataset_json_cache[dataset_code]
    json_path = os.path.join(path_base, dataset_code, f'{dataset_code}.json')
    if not os.path.isfile(json_path):
        print(f'[警告] 找不到 dataset JSON: {json_path}')
        _dataset_json_cache[dataset_code] = {}
        return {}
    with open(json_path, 'r', encoding='utf-8') as f:
        d = json.load(f)
    labels = d.get('labels', {})
    _dataset_json_cache[dataset_code] = labels
    return labels


def build_samples_from_m3d_seg(m3d_seg_root, dataset_codes, path_base=None):
    if path_base is None:
        path_base = os.path.join(m3d_seg_root, 'M3D_Seg')
    samples = []
    for dc in dataset_codes:
        json_path = os.path.join(path_base, dc, f'{dc}.json')
        if not os.path.isfile(json_path):
            print(f'[警告] JSON 不存在，跳过: {json_path}')
            continue
        with open(json_path, 'r', encoding='utf-8') as f:
            d = json.load(f)
        labels = d.get('labels', {})
        if not labels:
            continue
        for split in ('train', 'test'):
            for item in d.get(split, []):
                image_rel = item.get('image')
                if image_rel:
                    samples.append({'image': image_rel})
    return samples


samples = build_samples_from_m3d_seg(M3D_SEG_ROOT, DATASET_CODES, PATH_BASE)
print(f'样本数: {len(samples)}')
if samples:
    print('首条示例:', json.dumps(samples[0], ensure_ascii=False))

[警告] JSON 不存在，跳过: /BDSZ6/private/user/yxd/data/M3D/M3D-Seg/M3D_Seg/00010/00010.json
[警告] JSON 不存在，跳过: /BDSZ6/private/user/yxd/data/M3D/M3D-Seg/M3D_Seg/00016/00016.json
样本数: 3773
首条示例: {"image": "0000/5/image.npy"}


In [61]:
def get_exp_for_label(label_name, label2name, term_dictionary,
                      default_template='The {} in the MRI scan.'):
    if not label_name or str(label_name).lower() == 'background':
        return None
    key = label2name.get(label_name, label_name)
    if isinstance(key, list):
        key = key[0] if key else label_name
    key_str = str(key).strip() if key is not None else ''
    descs = (
        term_dictionary.get(key)
        or term_dictionary.get(key_str.lower() if key_str else '')
        or term_dictionary.get(label_name)
        or term_dictionary.get(str(label_name).lower() if label_name else '')
    )
    if isinstance(descs, list) and descs:
        return random.choice(descs)
    return default_template.format(key_str or str(label_name) or '')


label2name = {}
term_dictionary = {}
if LABEL2NAME_PATH and os.path.isfile(LABEL2NAME_PATH):
    with open(LABEL2NAME_PATH, 'r', encoding='utf-8') as f:
        label2name = json.load(f)
if TERM_DICTIONARY_PATH and os.path.isfile(TERM_DICTIONARY_PATH):
    with open(TERM_DICTIONARY_PATH, 'r', encoding='utf-8') as f:
        term_dictionary = json.load(f)
print(f'label2name entries: {len(label2name)},  term_dictionary entries: {len(term_dictionary)}')

label2name entries: 0,  term_dictionary entries: 223


## 3. 划分与目录初始化

In [ ]:
# ── 划分 train / test ─────────────────────────────────────────────────────────
# 按 video_id 字符串排序（而非原始 image_rel 路径）确定性划分，保证多次运行结果一致，
# 且与 load_refytvos_json 的 sorted(video_ids) 读取顺序一致。
#
# 修复原因：对于非零填充的数字 case_id（如数据集 0000 中的 1,2,...,10,11,...），
# 路径排序与字符串排序结果不同：
#   路径排序: 0000/1/ < 0000/10/  (因为 '/' ASCII 47 < '1' ASCII 49)
#   video_id 字符串排序: 0000_10_image < 0000_1_image  (因为 '0' ASCII 48 < '_' ASCII 95)
# 若用路径排序写 PKL，anno_id=0 对应 0000_1_image，但 rvos.py 读取时把 anno_id=0
# 分配给 0000_10_image，导致 mask 错位。
def _sort_key(s):
    image_rel = (s.get('image') or '').replace('\\', '/')
    vid = image_rel.replace('/', '_')
    if vid.lower().endswith('.npy'):
        vid = vid[:-4]
    return re.sub(r'[^\w\-]', '_', vid).strip('_')

sorted_indices = sorted(range(len(samples)), key=lambda i: _sort_key(samples[i]))

indices_all = list(range(len(samples)))
random.seed(SEED)
random.shuffle(indices_all)
n_train = max(1, int(len(indices_all) * TRAIN_RATIO))
train_set = set(indices_all[:n_train])
test_set  = set(indices_all[n_train:])
print(f'train: {len(train_set)},  test: {len(test_set)}')

# ── 目录树 ────────────────────────────────────────────────────────────────────
for split_dir in (TRAIN_DIR, TEST_DIR):
    for sub in ('JPEGImages', 'Annotations'):
        os.makedirs(os.path.join(split_dir, sub), exist_ok=True)
print('目录已创建:', OUTPUT_REVOS_DIR)

## 4. 主处理循环

In [63]:
# ── 每个 split 独立的累积结构 ─────────────────────────────────────────────────
#
# meta_for_loader   : {video_id: {objects: {obj_id: {category, frames}}}}   → meta.json
# all_metas         : {video_id: {expressions, frames, height, width}}      → meta_expressions.json
# mask_dict         : {str(anno_count): [rle_or_None × N_SLICES]}           → mask_dict.pkl
# anno_count        : per-split sequential counter (matches load_refytvos_json logic)
#
# anno_count must be incremented in sorted(video_ids) → sorted(expressions) order
# to stay consistent with how RVOSDataset / load_refytvos_json reads mask_dict.

data = {
    'train': {'meta': {}, 'metas_expr': {}, 'mask_dict': {}, 'anno_count': 0},
    'test':  {'meta': {}, 'metas_expr': {}, 'mask_dict': {}, 'anno_count': 0},
}

for rank, idx in enumerate(tqdm(sorted_indices, desc='Processing volumes')):
    s          = samples[idx]
    image_rel  = (s.get('image') or s.get('image_path') or '').replace('\\', '/')
    if not image_rel:
        continue

    split      = 'train' if idx in train_set else 'test'
    split_dir  = TRAIN_DIR if split == 'train' else TEST_DIR
    d          = data[split]

    dataset_code = image_rel.split('/')[0]
    labels_dict  = load_dataset_labels(PATH_BASE, dataset_code)
    if not labels_dict:
        continue

    image_path = os.path.join(PATH_BASE, image_rel)
    if not os.path.isfile(image_path):
        continue

    mask_path = get_mask_path_from_image_path(PATH_BASE, image_rel)
    if not mask_path or not os.path.isfile(mask_path):
        continue

    # video_id: e.g. '0000_5_image'
    video_id = image_rel.replace('/', '_')
    if video_id.lower().endswith('.npy'):
        video_id = video_id[:-4]
    video_id = re.sub(r'[^\w\-]', '_', video_id).strip('_') or f'vol_{idx}'

    # ── 加载图像切片 ──────────────────────────────────────────────────────────
    try:
        slices_list, slice_indices = load_3d_medical_image(
            image_path,
            n_slices=N_SLICES,
            axis=AXIS,
            sampling=SAMPLING,
            normalize=True,
            window_level=WINDOW_LEVEL,
            window_width=WINDOW_WIDTH,
        )
    except Exception as e:
        print(f'跳过 {image_path}: {e}')
        continue

    n_frames = len(slices_list)
    if n_frames == 0:
        continue

    frame_names = [f'{i:05d}' for i in range(n_frames)]
    height, width = slices_list[0].size[1], slices_list[0].size[0]

    # ── 写 JPEGImages ─────────────────────────────────────────────────────────
    out_jpeg_dir = os.path.join(split_dir, 'JPEGImages', video_id)
    os.makedirs(out_jpeg_dir, exist_ok=True)
    for fi, im in enumerate(slices_list):
        im.save(os.path.join(out_jpeg_dir, f'{frame_names[fi]}.jpg'), quality=95)

    # ── 加载 mask ─────────────────────────────────────────────────────────────
    try:
        gt_array, depth = load_mask_npz(mask_path)
    except Exception as e:
        print(f'Mask 加载失败 {mask_path}: {e}')
        continue
    if gt_array is None or depth == 0:
        continue

    # 预计算每帧对应的 z 索引（避免内层循环重复计算）
    z_indices = np.array([slice_indices[fi] if fi < len(slice_indices) else fi
                          for fi in range(n_frames)])

    # ── 遍历 labels，构建 expressions / Annotations / mask_dict ───────────────
    expressions_list = []   # [(exp_i, exp_text, obj_id_str)]
    object_infos     = []   # [(class_id, label_name)]
    masks_per_frame_all = {}  # {fi: [(obj_id, bbox_mask_2d), ...]}

    for label_key, label_name in sorted(labels_dict.items(), key=lambda x: int(x[0])):
        if str(label_name).strip().lower() == 'background':
            continue

        label_id = int(label_key)
        mask_idx = label_id - 1
        if mask_idx < 0 or mask_idx >= gt_array.shape[0]:
            print(f'[警告] mask_idx={mask_idx} 越界 ({gt_array.shape[0]})，跳过 {label_key}={label_name}')
            continue

        class_id = len(expressions_list)
        obj_id   = class_id + 1  # 1-based, consistent with meta.json objects key

        exp_text = get_exp_for_label(label_name, label2name, term_dictionary)
        if not exp_text:
            exp_text = f'The {label_name} in the MRI scan.'

        # ── Annotation PNG：累积单个 enclosing bbox mask per frame ───────────
        for fi in range(n_frames):
            m = mask_to_bbox_mask(mask_slice_at_z(gt_array, mask_idx, int(z_indices[fi])))
            masks_per_frame_all.setdefault(fi, []).append((obj_id, m))

        # ── mask_dict：bbox-mask RLE 列表（长度 = N_SLICES）──────────────────
        rle_masks = rle_list_from_gt_array(gt_array, mask_idx, slice_indices)
        anno_id_str = str(d['anno_count'])
        d['mask_dict'][anno_id_str] = rle_masks
        d['anno_count'] += 1

        expressions_list.append((class_id, exp_text, str(obj_id)))
        object_infos.append((class_id, str(label_name)))

    if not expressions_list:
        continue

    # ── 写 Annotations PNG ───────────────────────────────────────────────────
    out_ann_dir = os.path.join(split_dir, 'Annotations', video_id)
    os.makedirs(out_ann_dir, exist_ok=True)
    for fi in range(n_frames):
        ann_path   = os.path.join(out_ann_dir, f'{frame_names[fi]}.png')
        frame_masks = masks_per_frame_all.get(fi, [])
        save_indexed_annotation_frame(frame_masks, ann_path, height, width)

    # ── meta.json ─────────────────────────────────────────────────────────────
    d['meta'][video_id] = {
        'objects': {
            str(cid + 1): {'category': name, 'frames': frame_names}
            for cid, name in object_infos
        }
    }

    # ── meta_expressions.json ─────────────────────────────────────────────────
    expressions = {
        str(exp_i): {'exp': exp_text, 'obj_id': obj_id_str}
        for exp_i, exp_text, obj_id_str in expressions_list
    }
    d['metas_expr'][video_id] = {
        'expressions': expressions,
        'frames':      frame_names,
        'height':      height,
        'width':       width,
    }

print(f"\ntrain: {len(data['train']['metas_expr'])} videos, "
      f"{data['train']['anno_count']} expressions")
print(f"test : {len(data['test']['metas_expr'])} videos, "
      f"{data['test']['anno_count']} expressions")


Processing volumes:   0%|          | 0/3773 [00:00<?, ?it/s]

Processing volumes: 100%|██████████| 3773/3773 [6:30:52<00:00,  6.22s/it]   


train: 3018 videos, 113870 expressions
test : 755 videos, 27954 expressions


## 5. 保存每个 split 的 meta / mask_dict

In [64]:
for split, split_dir in (('train', TRAIN_DIR), ('test', TEST_DIR)):
    d = data[split]
    n_videos = len(d['metas_expr'])
    n_expr   = d['anno_count']
    print(f'\n── {split.upper()} ({n_videos} videos, {n_expr} expressions) ──')
    if n_videos == 0:
        print('  空 split，跳过。')
        continue

    # meta.json
    meta_path = os.path.join(split_dir, 'meta.json')
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump({'videos': d['meta']}, f, indent=2)
    print(f'  写入: {meta_path}')

    # meta_expressions.json
    expr_path = os.path.join(split_dir, 'meta_expressions.json')
    with open(expr_path, 'w', encoding='utf-8') as f:
        json.dump({'videos': d['metas_expr']}, f, ensure_ascii=False, indent=2)
    print(f'  写入: {expr_path}')

    # mask_dict.pkl
    pkl_path = os.path.join(split_dir, 'mask_dict.pkl')
    with open(pkl_path, 'wb') as f:
        pickle.dump(d['mask_dict'], f)
    size_mb = os.path.getsize(pkl_path) / 1024 / 1024
    n_valid = sum(1 for v in d['mask_dict'].values() if any(x is not None for x in v))
    print(f'  写入: {pkl_path}  ({size_mb:.1f} MB)')
    print(f'  mask_dict: {len(d["mask_dict"])} 条，其中有效（>=1 帧有前景）: {n_valid}')

print('\n完成。')


── TRAIN (3018 videos, 113870 expressions) ──
  写入: /BDSZ6/private/user/yxd/data/M3D/data_all/train/meta.json
  写入: /BDSZ6/private/user/yxd/data/M3D/data_all/train/meta_expressions.json
  写入: /BDSZ6/private/user/yxd/data/M3D/data_all/train/mask_dict.pkl  (214.3 MB)
  mask_dict: 113870 条，其中有效（>=1 帧有前景）: 71651

── TEST (755 videos, 27954 expressions) ──
  写入: /BDSZ6/private/user/yxd/data/M3D/data_all/train_eval/meta.json
  写入: /BDSZ6/private/user/yxd/data/M3D/data_all/train_eval/meta_expressions.json
  写入: /BDSZ6/private/user/yxd/data/M3D/data_all/train_eval/mask_dict.pkl  (52.6 MB)
  mask_dict: 27954 条，其中有效（>=1 帧有前景）: 18334

完成。


## 6. 快速校验

In [65]:
from pathlib import Path

def validate_split(split_dir, max_videos=20):
    base = Path(split_dir)
    meta_expr_path = base / 'meta_expressions.json'
    meta_path      = base / 'meta.json'
    pkl_path       = base / 'mask_dict.pkl'

    if not meta_expr_path.exists():
        print(f'  ✗ 找不到 {meta_expr_path}')
        return

    expr   = json.load(open(meta_expr_path))['videos']
    meta   = json.load(open(meta_path))['videos'] if meta_path.exists() else {}
    mdict  = pickle.load(open(pkl_path, 'rb')) if pkl_path.exists() else None

    print(f'  videos: {len(expr)},  mask_dict keys: {len(mdict) if mdict else 0}')

    problems = []
    # ── 重建 anno_count 顺序，验证 mask_dict key 一致性 ──────────────────────
    anno_count = 0
    for vid in sorted(expr.keys())[:max_videos]:
        frames   = sorted(expr[vid]['frames'])
        ann_dir  = base / 'Annotations' / vid
        jpeg_dir_v = base / 'JPEGImages' / vid

        if not ann_dir.exists():
            problems.append((vid, 'missing Annotations dir'))
        if not jpeg_dir_v.exists():
            problems.append((vid, 'missing JPEGImages dir'))

        for exp_id in sorted(expr[vid]['expressions'].keys(), key=int):  # integer sort
            key = str(anno_count)
            if mdict is not None and key not in mdict:
                problems.append((vid, f'mask_dict missing key {key} (exp {exp_id})'))
            elif mdict is not None:
                if len(mdict[key]) != len(frames):
                    problems.append((vid, f'key {key}: len={len(mdict[key])} != {len(frames)}'))
            anno_count += 1

        # PNG 帧存在性
        if ann_dir.exists():
            for fn in frames[:5]:   # 只检查前 5 帧
                if not (ann_dir / f'{fn}.png').exists():
                    problems.append((vid, f'missing {fn}.png'))

    if problems:
        print(f'  ⚠ {len(problems)} 个问题（前 10 条）:')
        for p in problems[:10]:
            print(f'    {p}')
    else:
        print(f'  ✓ 检查通过（前 {max_videos} 个视频）')


for split, split_dir in (('train', TRAIN_DIR), ('test', TEST_DIR)):
    print(f'\n=== {split.upper()} 校验 ===')
    if not os.path.exists(os.path.join(split_dir, 'meta_expressions.json')):
        print('  (空 split，跳过)')
        continue
    validate_split(split_dir, max_videos=20)



=== TRAIN 校验 ===
  videos: 3018,  mask_dict keys: 113870
  ✓ 检查通过（前 20 个视频）

=== TEST 校验 ===
  videos: 755,  mask_dict keys: 27954
  ✓ 检查通过（前 20 个视频）


## 7. 抽样内容核验（可选）

In [ ]:
# 抽取 train split 的第一个视频第一条 expression，对比 mask_dict RLE 与 Annotation PNG
import pycocotools.mask as coco_mask


def bbox_from_mask(mask: np.ndarray):
    """返回单个包围整张 mask 的 bbox: (x1, y1, x2, y2)；空 mask 返回 None。"""
    if mask is None:
        return None
    mask = np.asarray(mask) > 0
    if not mask.any():
        return None
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    y1, y2 = np.where(rows)[0][[0, -1]]
    x1, x2 = np.where(cols)[0][[0, -1]]
    return int(x1), int(y1), int(x2), int(y2)


for split, split_dir in (('train', TRAIN_DIR), ('test', TEST_DIR)):
    meta_expr_path = os.path.join(split_dir, 'meta_expressions.json')
    pkl_path       = os.path.join(split_dir, 'mask_dict.pkl')
    if not os.path.exists(meta_expr_path) or not os.path.exists(pkl_path):
        continue

    expr_data = json.load(open(meta_expr_path))['videos']
    mask_dict = pickle.load(open(pkl_path, 'rb'))

    vid       = sorted(expr_data.keys())[0]
    frames    = sorted(expr_data[vid]['frames'])
    exp_id    = sorted(expr_data[vid]['expressions'].keys(), key=int)[0]  # integer sort
    obj_id    = int(expr_data[vid]['expressions'][exp_id]['obj_id'])

    # mask_dict key = anno_count 顺序中的位置
    anno_count = 0
    target_key = None
    for v in sorted(expr_data.keys()):
        for eid in sorted(expr_data[v]['expressions'].keys(), key=int):  # integer sort
            if v == vid and eid == exp_id:
                target_key = str(anno_count)
                break
            anno_count += 1
        if target_key is not None:
            break

    rle_list = mask_dict.get(target_key, [])
    rle_valid = sum(1 for r in rle_list if r is not None)

    # 第一个有效帧
    first_valid_fi = next((i for i, r in enumerate(rle_list) if r is not None), None)
    if first_valid_fi is not None:
        rle_mask = coco_mask.decode(rle_list[first_valid_fi])
        png_path = os.path.join(split_dir, 'Annotations', vid, f'{frames[first_valid_fi]}.png')
        # 注意：Annotations PNG 以 mode='L' 保存，np.array() 直接返回像素索引值
        png_arr  = np.array(Image.open(png_path)) if os.path.exists(png_path) else None
        png_mask = (png_arr == obj_id).astype(np.uint8) if png_arr is not None else None
        match    = np.array_equal(rle_mask, png_mask) if png_mask is not None else 'N/A'

        rle_box = bbox_from_mask(rle_mask)
        png_box = bbox_from_mask(png_mask) if png_mask is not None else None
    else:
        rle_mask = png_mask = None
        rle_box = png_box = None
        match = 'no valid frame'

    print(f'
[{split}] video={vid}, exp_id={exp_id}, mask_dict_key={target_key}')
    print(f'  frames={len(frames)}, mask_dict RLE entries={len(rle_list)}, valid frames={rle_valid}')
    print(f'  RLE mask sum (frame {first_valid_fi}): {rle_mask.sum() if rle_mask is not None else "N/A"}')
    print(f'  PNG mask sum (obj_id={obj_id}): {png_mask.sum() if png_mask is not None else "N/A"}')
    print(f'  RLE vs PNG match: {match}')
    print(f'  RLE bbox: {rle_box}')
    print(f'  PNG bbox: {png_box}')


## 8. 可视化：PNG Annotations vs mask_dict.pkl 一致性对比

In [54]:
"""
可视化模块：比较 PNG Annotations 文件与 mask_dict.pkl RLE 中的 bbox-mask 和单个 bbox 是否一致。

对每个 split 各选取若干表达式，在同一图中并排显示：
  左列  — Annotation PNG 解码的 bbox-mask（蓝色半透明覆盖）+ 单个 bbox（蓝框）
  右列  — mask_dict.pkl RLE 解码的 bbox-mask（红色半透明覆盖）+ 单个 bbox（红框）

标题注明两者是否完全一致，以及当前帧是否存在 bbox。
"""

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pycocotools.mask as coco_mask

OUTPUT_REVOS_DIR  = '/BDSZ6/private/user/yxd/data/M3D/data_2'  # 根输出目录
TRAIN_DIR         = os.path.join(OUTPUT_REVOS_DIR, 'train')
TEST_DIR          = os.path.join(OUTPUT_REVOS_DIR, 'train_eval')
# ── 参数 ───────────────────────────────────────────────────────────────────────
N_VIDEOS_PER_SPLIT = 3   # 每个 split 抽取的视频数
N_FRAMES_PER_VIDEO = 4   # 每个视频显示的有效帧数（取前 N 个有前景的帧）
VIS_OUTPUT_DIR     = os.path.join(OUTPUT_REVOS_DIR, 'vis_consistency')
os.makedirs(VIS_OUTPUT_DIR, exist_ok=True)


# ── 工具函数（与 cell-verify 中 bbox_from_mask 保持一致）──────────────────────

def bbox_from_mask_vis(mask: np.ndarray):
    """返回单个包围整张 mask 的 bbox: (x1, y1, x2, y2)；空 mask 返回 None。"""
    if mask is None:
        return None
    mask = np.asarray(mask) > 0
    if not mask.any():
        return None
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    y1, y2 = np.where(rows)[0][[0, -1]]
    x1, x2 = np.where(cols)[0][[0, -1]]
    return int(x1), int(y1), int(x2), int(y2)


def draw_mask_and_boxes(ax, bg_img, mask, color_rgb, label_prefix, frame_idx):
    """在 ax 上绘制背景帧 + 半透明 bbox-mask 覆盖 + 单个 bbox。"""
    ax.imshow(bg_img, interpolation='bilinear')

    has_bbox = False
    if mask is not None and mask.any():
        overlay = np.zeros((*mask.shape, 4), dtype=np.float32)
        overlay[mask > 0] = [*[c / 255 for c in color_rgb], 0.45]
        ax.imshow(overlay)

        bbox = bbox_from_mask_vis(mask)
        if bbox is not None:
            x1, y1, x2, y2 = bbox
            rect = mpatches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=1.5,
                edgecolor=[c / 255 for c in color_rgb],
                facecolor='none',
            )
            ax.add_patch(rect)
            ax.text(x1 + 2, y1 + 2,
                    'bbox', fontsize=5,
                    color=[c / 255 for c in color_rgb],
                    verticalalignment='top',
                    bbox=dict(facecolor='black', alpha=0.4, pad=1, edgecolor='none'))
            has_bbox = True

    ax.set_title(f'{label_prefix}
frm={frame_idx}  bbox={"yes" if has_bbox else "no"}',
                 fontsize=7, pad=2)
    ax.axis('off')


def load_jpeg_frame(split_dir, video_id, frame_name):
    """加载 JPEGImages 帧，找不到则返回灰色 placeholder。"""
    for ext in ('jpg', 'jpeg', 'png'):
        p = os.path.join(split_dir, 'JPEGImages', video_id, f'{frame_name}.{ext}')
        if os.path.exists(p):
            return np.array(Image.open(p).convert('RGB'), dtype=np.float32) / 255.0
    return np.zeros((512, 512, 3), dtype=np.float32)


# ── 主循环 ─────────────────────────────────────────────────────────────────────

PNG_COLOR = (30, 144, 255)    # dodgerblue — PNG Annotations
PKL_COLOR = (220, 50,  50)    # red        — mask_dict.pkl RLE

summary_lines = []

for split, split_dir in (('train', TRAIN_DIR), ('test', TEST_DIR)):
    meta_expr_path = os.path.join(split_dir, 'meta_expressions.json')
    pkl_path       = os.path.join(split_dir, 'mask_dict.pkl')

    if not os.path.exists(meta_expr_path) or not os.path.exists(pkl_path):
        print(f'[{split}] 数据不存在，跳过。')
        continue

    expr_data = json.load(open(meta_expr_path))['videos']
    mask_dict = pickle.load(open(pkl_path, 'rb'))

    # 重建 anno_count_map — 必须用 key=int，与生成时的 integer sort 一致
    anno_count_map = {}
    cnt = 0
    for vid in sorted(expr_data.keys()):
        for eid in sorted(expr_data[vid]['expressions'].keys(), key=int):  # integer sort
            anno_count_map[(vid, str(eid))] = str(cnt)
            cnt += 1

    selected_vids = sorted(expr_data.keys())[:N_VIDEOS_PER_SPLIT]

    for vid in selected_vids:
        frames_all = sorted(expr_data[vid]['frames'])

        for exp_id in sorted(expr_data[vid]['expressions'].keys(), key=int):  # integer sort
            obj_id   = int(expr_data[vid]['expressions'][exp_id]['obj_id'])
            anno_key = anno_count_map.get((vid, str(exp_id)))
            if anno_key is None:
                continue

            rle_list = mask_dict.get(anno_key) or []

            # 收集有前景的帧（最多 N_FRAMES_PER_VIDEO 个）
            valid_frames = []
            for fi, rle in enumerate(rle_list):
                if rle is None:
                    continue
                m = coco_mask.decode(rle)
                if m.any():
                    valid_frames.append((fi, m))
                if len(valid_frames) >= N_FRAMES_PER_VIDEO:
                    break

            if not valid_frames:
                continue

            n_show = len(valid_frames)
            # 图：每帧 2 行（PNG | PKL），共 n_show 列 × 2 行
            fig, axes = plt.subplots(
                2, n_show,
                figsize=(n_show * 3.2, 7.0),
                squeeze=False,
            )
            fig.patch.set_facecolor('#111111')

            exp_text = expr_data[vid]['expressions'][exp_id]['exp']
            fig.suptitle(
                f'[{split}] {vid}  exp={exp_id}  obj_id={obj_id}
'
                f'anno_key={anno_key}  "{exp_text[:80]}"',
                color='white', fontsize=8, y=1.01,
            )

            all_match = True
            for col, (fi, pkl_mask) in enumerate(valid_frames):
                frame_name = frames_all[fi]
                bg_img = load_jpeg_frame(split_dir, vid, frame_name)

                # PNG Annotation → mask
                ann_png = os.path.join(split_dir, 'Annotations', vid, f'{frame_name}.png')
                if os.path.exists(ann_png):
                    png_arr  = np.array(Image.open(ann_png))
                    png_mask = (png_arr == obj_id).astype(np.uint8)
                else:
                    png_mask = None

                match = (np.array_equal(pkl_mask, png_mask)
                         if png_mask is not None else False)
                all_match = all_match and match

                # 顶行 = PNG, 底行 = PKL
                draw_mask_and_boxes(
                    axes[0, col], bg_img, png_mask,
                    PNG_COLOR,
                    f'PNG  {"✓match" if match else "✗MISMATCH"}',
                    fi,
                )
                draw_mask_and_boxes(
                    axes[1, col], bg_img, pkl_mask,
                    PKL_COLOR,
                    'PKL (RLE)',
                    fi,
                )

            # 行标签
            for row, label in enumerate(['PNG Annotation', 'PKL (RLE bbox-mask)']):
                axes[row, 0].set_ylabel(label, color='white', fontsize=9,
                                         rotation=90, labelpad=4)

            # 图例
            handles = [
                mpatches.Patch(color=[c/255 for c in PNG_COLOR], alpha=0.6,
                               label='PNG bbox-mask + single bbox'),
                mpatches.Patch(color=[c/255 for c in PKL_COLOR], alpha=0.6,
                               label='PKL RLE bbox-mask + single bbox'),
            ]
            fig.legend(handles=handles, loc='lower center', fontsize=8,
                       ncol=2, facecolor='#333333', labelcolor='white',
                       framealpha=0.8, bbox_to_anchor=(0.5, -0.04))

            plt.tight_layout(rect=[0, 0.04, 1, 1])

            out_name = f'{split}_{vid}_exp{exp_id}.png'
            out_path = os.path.join(VIS_OUTPUT_DIR, out_name)
            fig.savefig(out_path, dpi=110, bbox_inches='tight',
                        facecolor=fig.get_facecolor())
            plt.close(fig)

            status = '✓ ALL MATCH' if all_match else '✗ HAS MISMATCH'
            msg = (f'[{split}] {vid} exp={exp_id} obj_id={obj_id} '
                   f'valid_frames={n_show}  {status}  → {out_name}')
            print(msg)
            summary_lines.append(msg)

print(f'
可视化图像已保存到: {VIS_OUTPUT_DIR}')
print(f'
摘要（共 {len(summary_lines)} 条）:')
for l in summary_lines:
    print(' ', l)


[train] 0002_amos_0001_image exp=0 obj_id=1 valid_frames=4  ✓ ALL MATCH  → train_0002_amos_0001_image_exp0.png
[train] 0002_amos_0001_image exp=1 obj_id=2 valid_frames=4  ✓ ALL MATCH  → train_0002_amos_0001_image_exp1.png
[train] 0002_amos_0001_image exp=2 obj_id=3 valid_frames=4  ✓ ALL MATCH  → train_0002_amos_0001_image_exp2.png
[train] 0002_amos_0001_image exp=3 obj_id=4 valid_frames=4  ✓ ALL MATCH  → train_0002_amos_0001_image_exp3.png
[train] 0002_amos_0001_image exp=4 obj_id=5 valid_frames=4  ✓ ALL MATCH  → train_0002_amos_0001_image_exp4.png
[train] 0002_amos_0001_image exp=5 obj_id=6 valid_frames=4  ✓ ALL MATCH  → train_0002_amos_0001_image_exp5.png
[train] 0002_amos_0001_image exp=6 obj_id=7 valid_frames=4  ✓ ALL MATCH  → train_0002_amos_0001_image_exp6.png
[train] 0002_amos_0001_image exp=7 obj_id=8 valid_frames=4  ✓ ALL MATCH  → train_0002_amos_0001_image_exp7.png
[train] 0002_amos_0001_image exp=8 obj_id=9 valid_frames=4  ✓ ALL MATCH  → train_0002_amos_0001_image_exp8.png
[

KeyboardInterrupt: 